In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Construct relatedness density for the entry analysis

Prepare the relatedness-density control in SI S5. Run from `revision_notebooks/` after `../5.1_firm_agg.ipynb` and `../5.2_data4reg.ipynb`, using the 2019 PATSTAT release.

The entry sample consists of firm–CCMT class observations: the firm has at least ten patent families in 1995–2004 and no patents in that CCMT class during this base period. These conditions are `period == 1995`, `wt >= 10` and `num_pat_ccmt == 0`. Entry is measured by whether the firm patents in the class in 2005–2014.

The `person_id` column holds standardized applicant PSN IDs. Read the firm portfolios selected in notebook 5.1, which include firms with at least 100 patent families in total. All firms with patents in the base period contribute to the code shares used to calculate revealed comparative advantage (RCA).


In [ ]:
import duckdb
import numpy as np
import pandas as pd
from scipy import sparse


entry_data = FIRM_DATA / 'firm_green_data.parquet'
entry_portfolios = FIRM_DATA / 'firm_pat_cpc.parquet'
patstat_cpc = PATSTAT / 'tls225.parquet'
patstat_years = PATSTAT / 'tls201.parquet'
output = OUTPUTS / 'entry'
output.mkdir(parents=True, exist_ok=True)

Relatedness between a detailed CPC code and a CCMT class is the number of patent families containing both divided by the larger of their separate family counts. This equals the smaller of the two conditional proportions, as defined in SI S5.

A firm specializes in a code when its RCA is at least one. Relatedness density measures how much of the technology related to a CCMT class the firm is already specialized in. It is the relatedness-weighted sum of these specialization indicators, with weights summing to one. Density is missing when the firm has no patents in sections A–H or the CCMT class has no measured links to those codes.


In [ ]:
def relatedness_density(firm_counts, joint, code_totals, target_totals):
    """Relatedness from code co-occurrence; firm specialization uses RCA >= 1."""
    denominator = np.maximum(code_totals[:, None], target_totals[None, :])
    phi = np.divide(joint.toarray(), denominator,
                    out=np.zeros(joint.shape), where=denominator > 0)
    totals = np.asarray(firm_counts.sum(axis=1)).ravel()
    code_shares = np.asarray(firm_counts.sum(axis=0)).ravel()
    code_shares /= code_shares.sum()
    present = firm_counts.tocoo(copy=True)
    expected = totals[present.row] * code_shares[present.col]
    present.data = (present.data >= expected).astype(float)
    present = present.tocsr()
    present.eliminate_zeros()
    numerator = np.asarray(present @ phi)
    phi_sum = phi.sum(axis=0)
    density = np.divide(numerator, phi_sum, out=np.full_like(numerator, np.nan),
                        where=phi_sum > 0)
    density[totals == 0] = np.nan
    return density, phi

## Load the entry sample and firm portfolios


In [ ]:
data = pd.read_parquet(entry_data, columns=[
    'person_id', 'y02', 'period', 'wt', 'num_pat_ccmt', 'num_pat_ccmt_2', 'path', 'src'])
data = data.query('period == 1995 and wt >= 10 and num_pat_ccmt == 0').copy()
target_key = data.y02.str.replace(r'\s+', '', regex=True).str.upper()
targets = pd.Index(sorted(target_key.unique()))
con = duckdb.connect()
con.register('target_keys', pd.DataFrame({'target': targets}))
con.read_parquet(str(patstat_cpc)).create_view('patstat_cpc')
con.read_parquet(str(patstat_years)).create_view('patstat_years')
con.read_parquet(str(entry_portfolios)).create_view('portfolios')

## Count codes occurring together in 1995–2004

Take the minimum `earliest_filing_year` over all applications in each family, then select families from 1995–2004. Remove spaces from CPC symbols, convert them to uppercase, and retain full codes in sections A–H. Identify CCMT classes by the main group before `/`. Count each family/code and family/CCMT assignment once.


In [ ]:
# Minimum date is taken over ALL applications of each family before filtering.
con.execute('''CREATE TEMP TABLE baseline_families AS
    SELECT docdb_family_id FROM patstat_years GROUP BY docdb_family_id
    HAVING min(earliest_filing_year) BETWEEN 1995 AND 2004''')
con.execute(r'''CREATE TEMP TABLE normalized_cpc AS
    SELECT p.docdb_family_id, regexp_replace(upper(cpc_class_symbol), '\s+', '', 'g') AS code
    FROM patstat_cpc p JOIN baseline_families b USING(docdb_family_id)''')
con.execute('''CREATE TEMP TABLE baseline_codes AS
    SELECT DISTINCT docdb_family_id, code FROM normalized_cpc
    WHERE regexp_matches(code, '^[A-H][0-9]{2}[A-Z]')''')
con.execute('''CREATE TEMP TABLE baseline_targets AS
    SELECT DISTINCT docdb_family_id, target FROM normalized_cpc
    JOIN target_keys ON split_part(code, '/', 1) = target''')
code_counts = con.sql('SELECT code, count(*) AS n FROM baseline_codes GROUP BY code').df()
target_counts = con.sql('SELECT target, count(*) AS n FROM baseline_targets GROUP BY target').df()
joint_counts = con.sql('''SELECT code, target, count(*) AS n FROM baseline_codes
    JOIN baseline_targets USING(docdb_family_id) GROUP BY code, target''').df()

## Calculate firm specialization and relatedness density

Count distinct families in each firm's CPC portfolio. Use all portfolio codes when calculating RCA, including codes with zero measured relatedness to the CCMT classes.


In [ ]:
# Use the saved portfolio's firm-specific dates and >=100-family universe.
# Keep all such firms for the RCA reference, including firms outside the entry sample.
counts = con.sql(r'''SELECT person_id AS id,
    regexp_replace(upper(cpc), '\s+', '', 'g') AS code,
    count(DISTINCT docdb_family_id) AS n FROM portfolios
    WHERE period = 1995 AND regexp_matches(regexp_replace(upper(cpc), '\s+', '', 'g'),
        '^[A-H][0-9]{2}[A-Z]') GROUP BY id, code''').df()
# Portfolio-only codes remain in the RCA denominator even with zero measured relatedness.
codes = pd.Index(sorted(set(code_counts.code).union(counts.code)))
ids = pd.Index(sorted(set(counts.id).union(data.person_id)))
firm_counts = sparse.csr_matrix((counts.n.to_numpy(float),
    (ids.get_indexer(counts.id), codes.get_indexer(counts.code))), shape=(len(ids), len(codes)))
joint = sparse.csr_matrix((joint_counts.n.to_numpy(float),
    (codes.get_indexer(joint_counts.code), targets.get_indexer(joint_counts.target))),
    shape=(len(codes), len(targets)))
code_n = code_counts.set_index('code').n.reindex(codes, fill_value=0).to_numpy(float)
target_n = target_counts.set_index('target').n.reindex(targets, fill_value=0).to_numpy(float)
density, phi = relatedness_density(firm_counts, joint, code_n, target_n)
rows, cols = ids.get_indexer(data.person_id), targets.get_indexer(target_key)
data['proximity_density'] = density[rows, cols]
data['portfolio_observed'] = np.asarray(firm_counts.sum(axis=1)).ravel()[rows] > 0
data['target_profile_observed'] = np.asarray(joint.sum(axis=0)).ravel()[cols] > 0

## Save the firm–CCMT observations for notebook 10


In [ ]:
data.to_parquet(output / 'entry_density.parquet', index=False)
joint_counts['phi'] = phi[codes.get_indexer(joint_counts.code), targets.get_indexer(joint_counts.target)]
joint_counts.to_parquet(output / 'target_cpc_links.parquet', index=False)
data[['proximity_density']].describe().to_csv(output / 'density_summary.csv')
con.close()

data[['proximity_density', 'portfolio_observed', 'target_profile_observed']].describe()